# 04 — Exploratory Visualization

Quick matplotlib charts to **get a look at the data** before analysis. These render inline only — nothing is saved to `outputs/` (publication-ready social charts come later via `06-viz-social` / `chart_factory`).

All data pulled read-only from `migration_flows` in DuckDB. Charts use the IRS individuals measure for counts (near-census of filers) and ACS for the survey view; each cell notes which.

Views:
1. Net migration by state, 2023 (diverging bar)
2. National state-to-state movers over time (line)
3. Top origin→destination corridors, 2023 (ranked bar)
3b. Net directed pairs — which way each relationship flows + net-vs-gross scatter
4. One state's inflow vs outflow partners (paired bar)
4b. Do dollars move the same direction as people? (AGI per person by direction)
5. IRS vs ACS agreement (scatter)
6. ACS reliability over time (share of flows with MOE < 50%)

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import duckdb

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config
from src.clean_quality import run_sql

cfg = load_config('config.yaml')
# read-only so this notebook coexists with a DBCode connection / other kernels
con = duckdb.connect(cfg['settings']['duckdb_file'], read_only=True)

# light exploratory style (not the social preset)
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.axisbelow': True,
    'font.size': 10,
})
GREEN, RED, BLUE, GREY = '#16A34A', '#DC2626', '#2563EB', '#9CA3AF'
print('connected (read-only):', cfg['settings']['duckdb_file'])

## 1. Net migration by state, 2023
IRS individuals (people who came in) minus (people who left), state-to-state only. Green = net gain, red = net loss.

In [ ]:
net = run_sql("""
    WITH o AS (SELECT origin_state st, SUM(irs_individuals_out) gone FROM migration_flows
               WHERE year=2023 AND origin_type IN ('state','dc') GROUP BY 1),
         i AS (SELECT dest_state st, SUM(irs_individuals_in) came FROM migration_flows
               WHERE year=2023 AND dest_type IN ('state','dc') GROUP BY 1)
    SELECT COALESCE(i.st,o.st) AS state, came - gone AS net
    FROM i FULL OUTER JOIN o ON i.st=o.st
    ORDER BY net
""", con)

fig, ax = plt.subplots(figsize=(8, 11))
colors = [GREEN if v >= 0 else RED for v in net['net']]
ax.barh(net['state'], net['net'], color=colors)
ax.axvline(0, color='#333', linewidth=0.8)
ax.set_title('Net state-to-state migration, 2023 (IRS individuals)', fontweight='bold', loc='left')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.set_xlabel('net people (in − out)')
plt.tight_layout(); plt.show()

## 1b. Net AGI by state — the tax base view
Same net-migration idea, but in **dollars** instead of people: net adjusted gross income (arrivals' AGI minus departures' AGI). AGI is in thousands in the raw data; shown here in billions.

Also plots **AGI per net person** for the notable movers — if a state's net dollars-per-head is high, it's pulling in higher earners, not just more people. (Ratio is unstable for states with near-zero net people, so only large movers are shown.)

In [ ]:
net_agi = run_sql("""
    WITH o AS (SELECT origin_state st, SUM(irs_agi_out) a FROM migration_flows
               WHERE year=2023 AND origin_type IN ('state','dc') GROUP BY 1),
         i AS (SELECT dest_state st, SUM(irs_agi_in) a FROM migration_flows
               WHERE year=2023 AND dest_type IN ('state','dc') GROUP BY 1)
    SELECT COALESCE(i.st,o.st) AS state, (COALESCE(i.a,0)-COALESCE(o.a,0))/1e6 AS net_agi_b
    FROM i FULL OUTER JOIN o ON i.st=o.st ORDER BY net_agi_b
""", con)

fig, ax = plt.subplots(figsize=(8, 11))
colors = [GREEN if v >= 0 else RED for v in net_agi['net_agi_b']]
ax.barh(net_agi['state'], net_agi['net_agi_b'], color=colors)
ax.axvline(0, color='#333', linewidth=0.8)
ax.set_title('Net AGI migration by state, 2023 ($ billions)', fontweight='bold', loc='left')
ax.set_xlabel('net adjusted gross income ($B, in − out)')
plt.tight_layout(); plt.show()

In [ ]:
# AGI per net person: net dollars / net people (large movers only)
import numpy as np
cmp = run_sql("""
    WITH op AS (SELECT origin_state st, SUM(irs_individuals_out) p, SUM(irs_agi_out) a FROM migration_flows WHERE year=2023 AND origin_type IN ('state','dc') GROUP BY 1),
         ip AS (SELECT dest_state st, SUM(irs_individuals_in) p, SUM(irs_agi_in) a FROM migration_flows WHERE year=2023 AND dest_type IN ('state','dc') GROUP BY 1)
    SELECT COALESCE(ip.st,op.st) state,
           COALESCE(ip.p,0)-COALESCE(op.p,0) net_people,
           (COALESCE(ip.a,0)-COALESCE(op.a,0))*1000.0 net_agi_dollars
    FROM ip FULL OUTER JOIN op ON ip.st=op.st
""", con)
cmp = cmp[cmp['net_people'].abs() >= 20000].copy()
# Signed so direction matches meaning: RIGHT (+) = income arriving per net arrival,
# LEFT (-) = income leaving per net departure. Magnitude = |AGI| / |people|.
cmp['agi_per_net_person'] = (cmp['net_agi_dollars'].abs() / cmp['net_people'].abs()) * np.sign(cmp['net_people'])
cmp = cmp.sort_values('agi_per_net_person')
fig, ax = plt.subplots(figsize=(9, 6))
colors = [GREEN if v >= 0 else RED for v in cmp['net_people']]
ax.barh(cmp['state'], cmp['agi_per_net_person'], color=colors)
ax.axvline(0, color='#333', linewidth=0.8)
ax.set_title('AGI per net migrant, 2023 (states with |net| ≥ 20k people)', fontweight='bold', loc='left')
ax.set_xlabel('$ per net migrant  —  right: income arriving per net arrival · left: income leaving per net departure')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${abs(v)/1e3:,.0f}k'))
plt.tight_layout(); plt.show()
print('Reading: FL pulls ~$181k per net arrival (high earners in); IL loses ~$111k per net departure (high earners out).')

## 2. National state-to-state movers over time
ACS survey estimate of total interstate movers per year. Note the 2020 gap (ACS 1-year suspended for COVID).

In [ ]:
ts = run_sql("""
    SELECT year, SUM(acs_migrants) AS movers FROM migration_flows
    WHERE origin_type IN ('state','dc') AND dest_type IN ('state','dc')
    GROUP BY year ORDER BY year
""", con)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ts['year'], ts['movers'], color=BLUE, linewidth=2.5, marker='o')
ax.set_title('U.S. interstate movers per year (ACS)', fontweight='bold', loc='left')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v/1e6:.1f}M'))
ax.set_ylabel('movers'); ax.set_xlabel('year')
ax.annotate('2020: no ACS 1-year (COVID)', xy=(2020, ts['movers'].mean()),
            xytext=(2020, ts['movers'].min()*0.98), ha='center', fontsize=8, color=GREY)
plt.tight_layout(); plt.show()

## 3. Top origin→destination corridors, 2023
The 15 largest single directed flows between states (IRS individuals).

In [ ]:
corr = run_sql("""
    SELECT origin_state || ' → ' || dest_state AS corridor, irs_individuals_out AS people
    FROM migration_flows
    WHERE year=2023 AND origin_type IN ('state','dc') AND dest_type IN ('state','dc')
      AND irs_individuals_out IS NOT NULL
    ORDER BY people DESC LIMIT 15
""", con).sort_values('people')

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(corr['corridor'], corr['people'], color=BLUE)
ax.bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=8)
ax.set_title('Largest state-to-state corridors, 2023 (IRS individuals)', fontweight='bold', loc='left')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
plt.tight_layout(); plt.show()

## 3b. Net directed pairs — which way does each relationship flow?
Gross corridors double-count people swapping places (TX→CO ≈ 25k, CO→TX ≈ 22k, but the *net* is only ~3k toward TX). This collapses each unordered pair {A,B} into a single **net** flow and labels the winning direction. This is the truer picture of where people are actually relocating.

IRS individuals, 2023. `origin_state < dest_state` keeps one row per pair; direction is assigned to whichever side is larger.

In [ ]:
net_pairs = run_sql("""
    WITH s AS (
        SELECT origin_state a, dest_state b, irs_individuals_out n
        FROM migration_flows
        WHERE year=2023 AND origin_type IN ('state','dc') AND dest_type IN ('state','dc')
          AND irs_individuals_out IS NOT NULL
    ),
    paired AS (
        SELECT f.a, f.b, f.n AS a_to_b, COALESCE(r.n, 0) AS b_to_a
        FROM s f LEFT JOIN s r ON r.a = f.b AND r.b = f.a
        WHERE f.a < f.b                       -- one row per unordered pair
    )
    SELECT
        CASE WHEN a_to_b >= b_to_a THEN a || ' → ' || b ELSE b || ' → ' || a END AS net_direction,
        ABS(a_to_b - b_to_a)  AS net_flow,
        (a_to_b + b_to_a)     AS gross_flow
    FROM paired
    ORDER BY net_flow DESC
    LIMIT 15
""", con).sort_values('net_flow')

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(net_pairs['net_direction'], net_pairs['net_flow'], color=BLUE)
ax.bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=8)
ax.set_title('Largest NET directed migrations, 2023 (IRS individuals)', fontweight='bold', loc='left')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.set_xlabel('net people moved (winning direction)')
plt.tight_layout(); plt.show()

### Net vs gross: why every point sits below the line
Each dot is a state pair. **X = gross flow** (everyone who moved either direction, A→B + B→A). **Y = net flow** (the directional imbalance, |A→B − B→A|).

The dashed diagonal is the **ceiling** `net = gross` — the most net flow a pair *could* have. A point can only reach it if the movement is purely one-way (nobody goes the other way). Since almost every pair has people moving both directions, **net is always ≤ gross, so every point sits on or below the line.** That's math, not a finding.

The real signal is *how far below* the line a point sits: the lower it is, the more the two directions cancel out. Across all pairs the median net is only ~9% of gross — meaning **about 91% of interstate movement is offset by someone moving the opposite way.** TX↔CO is the classic high-churn swap (big gross, tiny net); the few points hugging the line are near one-way drains.

In [ ]:
nv = run_sql("""
    WITH s AS (
        SELECT origin_state a, dest_state b, irs_individuals_out n
        FROM migration_flows
        WHERE year=2023 AND origin_type IN ('state','dc') AND dest_type IN ('state','dc')
          AND irs_individuals_out IS NOT NULL
    ),
    paired AS (
        SELECT f.a, f.b, f.n AS a_to_b, COALESCE(r.n, 0) AS b_to_a
        FROM s f LEFT JOIN s r ON r.a = f.b AND r.b = f.a
        WHERE f.a < f.b
    )
    SELECT a, b, ABS(a_to_b - b_to_a) AS net_flow, (a_to_b + b_to_a) AS gross_flow,
           ABS(a_to_b - b_to_a) * 1.0 / NULLIF(a_to_b + b_to_a, 0) AS net_ratio
    FROM paired
""", con)

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(nv['gross_flow'], nv['net_flow'], s=14, alpha=0.4, color=BLUE)
m = nv['gross_flow'].max()
ax.plot([0, m], [0, m], '--', color=GREY, linewidth=1, label='ceiling: net = gross (pure one-way)')
med = nv['net_ratio'].median()
ax.plot([0, m], [0, m*med], ':', color=RED, linewidth=1.5, label=f'median pair: net = {med:.0%} of gross')
# label the most extreme pairs by net_flow
for _, r in nv.nlargest(6, 'net_flow').iterrows():
    ax.annotate(f"{r['a']}–{r['b']}", (r['gross_flow'], r['net_flow']),
                fontsize=8, xytext=(4, 2), textcoords='offset points')
ax.set_xlabel('gross flow (both directions)'); ax.set_ylabel('net flow (imbalance)')
ax.set_title('Net vs gross state-pair flows, 2023 — most movement cancels out', fontweight='bold', loc='left')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v/1e3:.0f}k'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v/1e3:.0f}k'))
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. One state's inflow vs outflow partners
Set `STATE` to any two-letter code. Shows its top partners in each direction (IRS individuals, 2023).

In [ ]:
STATE = 'CA'   # <- change me
N = 10

outp = run_sql(f"""
    SELECT dest_state AS partner, irs_individuals_out AS people FROM migration_flows
    WHERE year=2023 AND origin_state='{STATE}' AND dest_type IN ('state','dc')
      AND irs_individuals_out IS NOT NULL ORDER BY people DESC LIMIT {N}
""", con)
inp = run_sql(f"""
    SELECT origin_state AS partner, irs_individuals_in AS people FROM migration_flows
    WHERE year=2023 AND dest_state='{STATE}' AND origin_type IN ('state','dc')
      AND irs_individuals_in IS NOT NULL ORDER BY people DESC LIMIT {N}
""", con)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5), sharex=False)
a1.barh(inp['partner'][::-1], inp['people'][::-1], color=GREEN)
a1.set_title(f'Top origins INTO {STATE}', fontweight='bold', loc='left')
a2.barh(outp['partner'][::-1], outp['people'][::-1], color=RED)
a2.set_title(f'Top destinations OUT of {STATE}', fontweight='bold', loc='left')
for a in (a1, a2):
    a.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
plt.tight_layout(); plt.show()

## 4b. Do dollars move the same direction as people?
For a pair, compare **net people** vs **net AGI** direction, plus the average AGI-per-person each way. If the per-person income differs by direction, that's the first hint of an income-selection story (e.g., higher earners leaving than arriving).

AGI is in thousands of dollars. Set `A` / `B` to any two state codes.

> Caveat: this is the *average* AGI per filer per direction — it can't see true AGI **brackets** (who is rich vs. poor within a flow). That needs the IRS AGI-bracket migration files, which are not in this dataset. Flag for `05-analysis` if bracket detail matters.

In [ ]:
A, B = 'CA', 'TX'   # <- change me

pair = run_sql(f"""
    SELECT origin_state, dest_state,
           irs_individuals_out AS people,
           irs_agi_out         AS agi_k,
           ROUND(irs_agi_out * 1000.0 / NULLIF(irs_individuals_out,0), 0) AS agi_per_person
    FROM migration_flows
    WHERE year=2023
      AND ((origin_state='{A}' AND dest_state='{B}') OR (origin_state='{B}' AND dest_state='{A}'))
""", con)
print(pair.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
labels = [f"{r.origin_state}→{r.dest_state}" for r in pair.itertuples()]
ax1.bar(labels, pair['people'], color=[BLUE, RED])
ax1.set_title(f'People moved, {A}↔{B} 2023', fontweight='bold', loc='left')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax2.bar(labels, pair['agi_per_person'], color=[BLUE, RED])
ax2.set_title('Avg AGI per person ($)', fontweight='bold', loc='left')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))
plt.tight_layout(); plt.show()

## 5. Do IRS and ACS agree on flow size?
Each dot is one state-to-state edge in 2023. IRS individuals (x) vs ACS migrants (y). The dashed line is y = x. Systematic offset is expected (ACS counts all residents; IRS counts tax filers), but the shape tells you how consistent the two sources are.

In [ ]:
cmp = run_sql("""
    SELECT irs_individuals_out AS irs, acs_migrants AS acs FROM migration_flows
    WHERE year=2023 AND origin_type IN ('state','dc') AND dest_type IN ('state','dc')
      AND irs_individuals_out IS NOT NULL AND acs_migrants IS NOT NULL AND acs_migrants > 0
""", con)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(cmp['irs'], cmp['acs'], s=10, alpha=0.4, color=BLUE)
lim = max(cmp['irs'].max(), cmp['acs'].max())
ax.plot([0, lim], [0, lim], '--', color=GREY, linewidth=1)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('IRS individuals (log)'); ax.set_ylabel('ACS migrants (log)')
ax.set_title('IRS vs ACS flow size, 2023', fontweight='bold', loc='left')
corr_val = cmp['irs'].corr(cmp['acs'])
ax.text(0.05, 0.95, f'Pearson r = {corr_val:.3f}', transform=ax.transAxes, va='top', fontsize=9)
plt.tight_layout(); plt.show()

## 6. ACS reliability over time
Share of state-to-state ACS flows where the margin of error is under 50% of the estimate. Small flows are noisy, so this is a reminder to lean on IRS (or filter by `acs_reliable`) for the long tail.

In [ ]:
rel = run_sql("""
    SELECT year,
           AVG(CASE WHEN acs_reliable THEN 1.0 ELSE 0.0 END) AS reliable_share
    FROM migration_flows
    WHERE acs_migrants IS NOT NULL
      AND origin_type IN ('state','dc') AND dest_type IN ('state','dc')
    GROUP BY year ORDER BY year
""", con)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(rel['year'].astype(str), rel['reliable_share'], color=BLUE)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_title('Share of ACS state-to-state flows that are reliable (MOE < 50%)', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

## 7. County income-class migration (2023)
Now at the county level, with each county flagged **above/below the national median AGI-per-return** (a resident-income proxy from non-migrant tax returns). This is the closest this data gets to your income-selection question — not the income of individual movers, but whether people flow between richer and poorer *places*.

> Caveat: county income class ≠ mover income. IRS does not publish migration by income bracket; this tags the *places*, not the *people* in each flow.

In [ ]:
# 7a. Flow matrix: people moving between county income classes (gross, IRS individuals)
mat = run_sql("""
    SELECT origin_income_class, dest_income_class, SUM(irs_individuals_out) ppl
    FROM county_migration_flows WHERE flow_class IS NOT NULL
    GROUP BY 1,2
""", con)
pivot = mat.pivot(index='origin_income_class', columns='dest_income_class', values='ppl')
pivot = pivot.reindex(index=['above_median','below_median'], columns=['above_median','below_median'])

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(pivot.values, cmap='Blues')
ax.set_xticks([0,1]); ax.set_xticklabels(['to above','to below'])
ax.set_yticks([0,1]); ax.set_yticklabels(['from above','from below'])
for r in range(2):
    for cc in range(2):
        ax.text(cc, r, f'{pivot.values[r,cc]:,.0f}', ha='center', va='center',
                color='white' if pivot.values[r,cc] > pivot.values.max()*0.5 else '#111', fontsize=11)
ax.set_title('County-to-county moves by income class, 2023 (people)', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

In [ ]:
# 7b. Net migration by county income class (in - out)
net_cls = run_sql("""
    WITH o AS (SELECT origin_fips f, SUM(irs_individuals_out) gone FROM county_migration_flows GROUP BY 1),
         i AS (SELECT dest_fips f, SUM(irs_individuals_in) came FROM county_migration_flows GROUP BY 1)
    SELECT ic.income_class,
           SUM(COALESCE(came,0) - COALESCE(gone,0)) AS net
    FROM county_income_class ic
    LEFT JOIN o ON o.f = ic.fips LEFT JOIN i ON i.f = ic.fips
    GROUP BY 1
""", con)

fig, ax = plt.subplots(figsize=(7, 4))
colors = [GREEN if v >= 0 else RED for v in net_cls['net']]
ax.bar(net_cls['income_class'], net_cls['net'], color=colors, width=0.5)
ax.axhline(0, color='#333', linewidth=0.8)
ax.set_title('Net county migration by income class, 2023', fontweight='bold', loc='left')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:,.0f}'))
ax.set_ylabel('net people (in − out)')
for i,v in enumerate(net_cls['net']):
    ax.text(i, v, f'{v:+,.0f}', ha='center', va='bottom' if v>=0 else 'top', fontsize=9)
plt.tight_layout(); plt.show()
print('Net positive for below-median counties = on net, people moved from richer to poorer counties.')

In [ ]:
# 7c. Largest county-to-county corridors, 2023 (IRS individuals)
cty_corr = run_sql("""
    SELECT origin_name || ' → ' || dest_name AS corridor,
           origin_income_class, dest_income_class, irs_individuals_out AS people
    FROM county_migration_flows
    WHERE irs_individuals_out IS NOT NULL
    ORDER BY people DESC LIMIT 15
""", con).sort_values('people')

fig, ax = plt.subplots(figsize=(10, 7))
# color by whether destination is above (blue) or below (orange) median
bar_colors = [BLUE if dc=='above_median' else '#D97706' for dc in cty_corr['dest_income_class']]
bars = ax.barh(cty_corr['corridor'], cty_corr['people'], color=bar_colors)
ax.bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=8)
ax.set_title('Largest county-to-county corridors, 2023', fontweight='bold', loc='left')
ax.set_xlabel('people (IRS individuals) — blue = to above-median county, orange = to below-median')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:,.0f}'))
plt.tight_layout(); plt.show()

In [ ]:
# 7d. Which counties gained/lost the most people, tagged by income class (2023)
cty_net = run_sql("""
    WITH o AS (SELECT origin_fips f, SUM(irs_individuals_out) gone FROM county_migration_flows GROUP BY 1),
         i AS (SELECT dest_fips f, SUM(irs_individuals_in) came FROM county_migration_flows GROUP BY 1)
    SELECT ic.name, ic.income_class,
           COALESCE(came,0) - COALESCE(gone,0) AS net
    FROM county_income_class ic
    LEFT JOIN o ON o.f=ic.fips LEFT JOIN i ON i.f=ic.fips
""", con)
top = cty_net.nlargest(12, 'net'); bot = cty_net.nsmallest(12, 'net')
combo = pd.concat([bot, top]).sort_values('net')

fig, ax = plt.subplots(figsize=(9, 8))
colors = [GREEN if v>=0 else RED for v in combo['net']]
ax.barh(combo['name'], combo['net'], color=colors)
ax.axvline(0, color='#333', linewidth=0.8)
ax.set_title('Biggest county net gainers & losers, 2023 (IRS individuals)', fontweight='bold', loc='left')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:,.0f}'))
plt.tight_layout(); plt.show()

---
These are throwaway exploration charts. Once you know which stories hold up in `05-analysis`, the keepers get rebuilt as branded social PNGs in `06-viz-social`.

```python
con.close()
```

---
## Cleanup
Close the DuckDB connection so the lock is released for other tools (DBCode, other notebooks). Runs on “Run All”.

In [ ]:
con.close()
print('connection closed')